# Mushroom Species Classification from Field Photographs
**Research Question:** How does model complexity affect the classification of mushroom species from field photographs?

**Sub-RQ1:** At what level of complexity do classifiers achieve reliable species identification?

**Sub-RQ2:** How can dangerous misclassifications between toxic species be minimized?

## 1. Setup and Data Loading

### Data Preparation (performed prior to this notebook)

The dataset was collected from iNaturalist Denmark (19 species, 500 images each).
Quality filtering removed 342 images (blur score < 50, extreme aspect ratios, very small images).
An observer-aware train/val/test split was performed to prevent data leakage from
photographers appearing across splits. See the report methodology for full details.

The filtered images and split files are provided in the repository.

In [1]:
# --- Configuration ---
RETRAIN = False  # Set True to retrain all models (~2 hours, GPU required)

# --- Download images if not present ---
import os
if not os.path.exists('data/images'):
    print('Downloading images from GitHub Releases...')
    !wget -q https://github.com/vinb75/mushroom-classification/releases/download/v1.0/images.zip
    !unzip -q images.zip -d data/
    !rm images.zip
    print('Done.')
else:
    print('Images already present.')

zsh:1: command not found: wget
unzip:  cannot find or open images.zip, images.zip.zip or images.zip.ZIP.
rm: images.zip: No such file or directory
Done.


In [ ]:
# --- Imports ---
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import time

from PIL import Image
from tqdm.auto import tqdm

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings('ignore')
tf.keras.utils.set_random_seed(42)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# --- Load split CSVs ---
SPLITS_DIR = 'data/splits'
IMAGES_DIR = 'data/images'

train_df = pd.read_csv(os.path.join(SPLITS_DIR, 'train.csv'))
val_df   = pd.read_csv(os.path.join(SPLITS_DIR, 'val.csv'))
test_df  = pd.read_csv(os.path.join(SPLITS_DIR, 'test.csv'))

# Add split labels and combine for EDA
train_df['split'] = 'train'
val_df['split']   = 'val'
test_df['split']  = 'test'
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# Absolute paths for image loading
all_df['abs_path'] = all_df['file_path'].apply(lambda p: os.path.join(os.getcwd(), p))

SPECIES   = sorted(all_df['label'].unique())
N_SPECIES = len(SPECIES)

print(f'{N_SPECIES} species, {len(all_df):,} total images')
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

---
## 2. Exploratory Data Analysis

PORT EDA CELLS HERE:
- 2.1 Class distribution (eda.ipynb cell 4)
- 2.2 Sample image grid (eda.ipynb cell 6)
- 2.3 Resolution distribution (eda.ipynb cells 8-9)
- 2.4 Quality audit (eda.ipynb cells 11-13)
- 2.5 Observer bias analysis (eda.ipynb cells 17-21)
- 2.6 PCA feature space exploration (eda.ipynb cells 30-32)
- 2.7 Pairwise species similarity (eda.ipynb cells 35-38)
- 2.8 Post-split verification (eda.ipynb cells 41-42)

When porting, replace:
- PROJECT_ROOT references → use os.getcwd()
- IMAGES_DIR → 'data/images'
- SPLITS_DIR → 'data/splits'
- abs_path function → already defined above

---
## 3. Feature Extraction for Classical Models

In [ ]:
# --- Load images and flatten to pixel feature vectors ---
# Adapted from modelling_baseline.ipynb

IMG_SIZE_PCA = 64  # small size for pixel-based features — keeps PCA tractable

def load_images_flat(df, img_size=IMG_SIZE_PCA):
    """
    Load images from a DataFrame, resize to img_size x img_size,
    and flatten each image into a 1D vector of pixel values.
    A 64x64 RGB image becomes a vector of 64*64*3 = 12,288 features.
    """
    images = []
    labels = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Loading images'):
        try:
            img = Image.open(row['file_path']).convert('RGB').resize((img_size, img_size))
            images.append(np.array(img).flatten())
            labels.append(row['label'])
        except Exception:
            continue
    return np.array(images), np.array(labels)

X_train_raw, y_train_raw = load_images_flat(train_df)
X_val_raw, y_val_raw     = load_images_flat(val_df)
X_test_raw, y_test_raw   = load_images_flat(test_df)

print(f'Train: {X_train_raw.shape[0]} images, {X_train_raw.shape[1]} features each')
print(f'Val:   {X_val_raw.shape[0]} images')
print(f'Test:  {X_test_raw.shape[0]} images')

In [ ]:
# --- Standardise and apply PCA ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled   = scaler.transform(X_val_raw)
X_test_scaled  = scaler.transform(X_test_raw)

N_COMPONENTS = 100
pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca   = pca.transform(X_val_scaled)
X_test_pca  = pca.transform(X_test_scaled)

variance_explained = pca.explained_variance_ratio_.sum()
print(f'PCA: {N_COMPONENTS} components explain {variance_explained*100:.1f}% of variance')

---
## 4. Main Model 1: Logistic Regression

In [ ]:
# --- Train logistic regression ---
# Adapted from modelling_baseline.ipynb

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_pca, y_train_raw)

lr_train_acc = lr_model.score(X_train_pca, y_train_raw)
lr_val_acc   = lr_model.score(X_val_pca, y_val_raw)
lr_test_acc  = lr_model.score(X_test_pca, y_test_raw)

print(f'Logistic Regression:')
print(f'  Train accuracy: {lr_train_acc*100:.1f}%')
print(f'  Val accuracy:   {lr_val_acc*100:.1f}%')
print(f'  Test accuracy:  {lr_test_acc*100:.1f}%')
print(f'  Random chance:  {100/N_SPECIES:.1f}%')

y_pred_lr = lr_model.predict(X_test_pca)
print('\nClassification Report:\n')
print(classification_report(y_test_raw, y_pred_lr, zero_division=0))

---
## 5. Main Model 2: SVM

TODO: Add SVM on PCA features

In [ ]:
# --- Train SVM ---
# TODO: implement

---
## 6. Main Model 3: Random Forest

TODO: Add Random Forest on PCA features

In [ ]:
# --- Train Random Forest ---
# TODO: implement

---
## 7. Main Model 4: Custom CNN

TODO: Build and train a small custom CNN from scratch

In [ ]:
# --- Custom CNN ---
# TODO: implement

---
## 8. Benchmark Baseline: Pretrained CNN (EfficientNet-B0)

Following the project guidelines, we use a pretrained CNN as a benchmark baseline
to establish an upper bound on classification performance for this task.

In [ ]:
# --- tf.data pipeline for CNN models ---
# Adapted from CNN_models_train_eval notebook

IMG_SIZE = 224
BATCH_SIZE = 16

# Encode labels for tf.data
le = LabelEncoder()
le.fit(train_df['label'])
train_df['label_id'] = le.transform(train_df['label'])
val_df['label_id']   = le.transform(val_df['label'])
test_df['label_id']  = le.transform(test_df['label'])
y_test = test_df['label_id'].values

def load_image(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)  # keep 0-255 range; model-specific preprocessing applied later
    return img, label

def make_dataset(df, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (df['file_path'].values, df['label_id'].values)
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(df), 1000), seed=42)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, shuffle=True)
val_ds   = make_dataset(val_df, shuffle=False)
test_ds  = make_dataset(test_df, shuffle=False)

print(f'Datasets built: {len(train_df)} train, {len(val_df)} val, {len(test_df)} test')

In [ ]:
# --- PORT: Helper functions from CNN notebook cells 10-11 ---
# plot_training_history, evaluate_model, build_and_train
# TODO: paste from CNN_models_train_eval cells 10 and 11

In [ ]:
# --- Train or load EfficientNet-B0 ---
if RETRAIN:
    efficientnet_model, efficientnet_results = build_and_train(
        base_model_fn=keras.applications.EfficientNetB0,
        model_name='EfficientNet-B0',
        preprocess_fn=keras.applications.efficientnet.preprocess_input,
    )
    efficientnet_model.save('models/efficientnet_b0.keras')
else:
    efficientnet_model = tf.keras.models.load_model('models/efficientnet_b0.keras')
    print('Loaded EfficientNet-B0 from disk')

---
## 9. Model Comparison

TODO: Build comparison table across all models
- Logistic Regression, SVM, Random Forest, Custom CNN, EfficientNet-B0
- Accuracy vs. model complexity scatter plot

---
## 10. Explainability: Grad-CAM Analysis

TODO: Port Grad-CAM cells from CNN notebook (cells 39-43)
- Correctly classified panel
- Misclassified panel
- Saturation note

---
## 11. Toxicity Cost Analysis

TODO: Port toxicity analysis
- Load toxicity_table.csv
- Apply cost matrix to predictions from each model
- Cross-model comparison

---
## 12. Confusion Matrix Cross-Reference with EDA

TODO: Port confusion analysis
- Top-5 confused pairs from B0
- Compare with EDA cosine-similarity predictions

---
## 13. Save Results

In [ ]:
# --- Save all results ---
os.makedirs('results', exist_ok=True)
# TODO: save comparison table, predictions CSVs, figures